# Lab 2 — Protect, Classify and Enrich Medicare Appeals with Cortex Code
### Triple-S Snow Day · Technical Track · 90 minutes

**Audience:** Data Engineers, BI Analysts, IT Developers and Healthcare Analysts who write SQL.

---

## The business problem

When Triple-S denies a Medicare Advantage service, the member can appeal (a *reconsideration*). CMS sets strict deadlines:

| Appeal type | Plan decision deadline (pre-service) |
|---|---|
| **Expedited** | **72 hours** |
| Standard | 30 days |

If the plan **upholds** the denial, or **misses the deadline**, the case is automatically forwarded to the **Independent Review Entity (IRE)**. When the IRE **overturns** the plan, that is a strong signal of systematic incorrect denials, and it shows up in Star Ratings and audits.

Triple-S already stores plan and IRE decisions, but:

1. **Nobody measures the IRE overturn rate by appeal reason**, because the reason is only in **free text**.
2. That free text contains **PHI** (names, dates of birth, Medicare IDs, phone numbers), so analysts can't use it until it is protected.
3. The categories that people enter by hand are **sometimes wrong**.

## What you will build: a medallion pipeline with AI inside
```
BRONZE_APPEALS  ──►  SILVER_APPEALS          ──►  ENRICHED_APPEALS                 ──►  GOLD views
 (raw, has PHI)      (masking + AI_REDACT)        (AI_CLASSIFY, AI_FILTER, AI_EXTRACT)    (SLA, overturn rates)
                     Dynamic Table, INCREMENTAL   Dynamic Table, INCREMENTAL
```

**Core technical lesson:** AI functions running inside **incremental Dynamic Tables** only process **new and changed rows**. That is what makes AI on every appeal affordable at scale.

**Core Cortex Code lesson:** you will *prototype* each AI step on a few rows with Cortex Code, **validate** the output, and only then let Cortex Code turn it into a production pipeline.

> **All data in this lab is synthetic.** Every name, date of birth, Medicare ID (MBI) and phone number is fake.

## Agenda (90 min)

| Time | Section | Cortex Code skill |
|---|---|---|
| 0–8 | **0. Setup + Cortex Code primer** | Context-rich prompts |
| 8–15 | **1. Explore BRONZE and estimate AI cost** | Ask before you spend |
| 15–30 | **2. Protect PHI: masking + AI_REDACT** | Prototype on 5 rows, **validate**, then productionize |
| 30–45 | **3. Classify and enrich** | Iterate on prompts and label sets |
| 45–55 | **4. Validate AI against humans** | Ask Cortex Code to audit the results |
| 55–72 | **5. Incremental AI with Dynamic Tables** | Have Cortex Code explain refresh behavior |
| 72–87 | **6. GOLD: SLA and overturn metrics** | Ask for metrics *and* a chart |
| 87–90 | Wrap-up | |

---
# Section 0 — Setup and Cortex Code primer

**Cortex Code** is the AI coding agent built into Snowflake. Open it with the **Cortex Code icon in the lower-right corner** of Snowsight. It can see this notebook, your objects and query history. It can write SQL and Python, run it, read errors and fix them.

### How productive engineers prompt Cortex Code
| Habit | Weak prompt | Strong prompt |
|---|---|---|
| **Give context** | *"Redact the text"* | *"In BRONZE_APPEALS, APPEAL_REASON_TEXT has PHI. Redact names, DOB, MBI and phone numbers with AI_REDACT."* |
| **Plan first** | *"Build the pipeline"* | *"Propose a Dynamic Table design for SILVER and ENRICHED that refreshes incrementally. Don't run anything yet."* |
| **Prototype small** | run on 300 rows | *"Test it on 5 rows first and show me input next to output."* |
| **Make it verify** | trust the output | *"Check if any MBI or phone pattern survived the redaction."* |
| **Ask why** | | *"Why did this Dynamic Table fall back to FULL refresh?"* |

> In every section you will see a **Cortex Code prompt** box followed by **reference cells**. Use Cortex Code first. The reference cells are the answer key and keep you on track if you fall behind.

## 0.1 Create your schema and load BRONZE
This creates `TSS_SNOWDAY.LAB2_<YOUR_USER>` and loads **300 synthetic appeals** into `BRONZE_APPEALS`, plus a held-back batch of **20 new appeals** in `BRONZE_APPEALS_NEW_BATCH` that we will use to show incremental processing in Section 5.

Safe to re-run (about 10 seconds).

In [ ]:
import re
from snowflake.snowpark.context import get_active_session

session = get_active_session()

db = "TSS_SNOWDAY"
user = session.sql("SELECT CURRENT_USER()").collect()[0][0]
schema = "LAB2_" + re.sub(r"[^A-Z0-9_]", "_", user.upper())
fq = f"{db}.{schema}"
wh = session.get_current_warehouse()

session.sql(f"CREATE DATABASE IF NOT EXISTS {db}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {fq}").collect()
session.use_schema(fq)

print(f"Your lab schema : {fq}")
print(f"Warehouse       : {wh}")

In [ ]:
LOAD_STATEMENTS = [
    """
CREATE OR REPLACE TRANSIENT TABLE __FQ__._APPEAL_GEN AS
WITH b AS (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 320)))
SELECT n,
       MOD(ABS(HASH(n, 'cat')), 6)                        AS cat,
       MOD(ABS(HASH(n, 'typ')), 100) < 40                 AS is_expedited,
       MOD(ABS(HASH(n, 'weak')), 100) < 35                AS is_weak,
       MOD(ABS(HASH(n, 'lang')), 100) < 7                 AS is_spanish,
       MOD(ABS(HASH(n, 'mislabel')), 100) < 12            AS is_mislabeled,
       MOD(ABS(HASH(n, 'late')), 100) < 12                AS is_late,
       (n > 300) OR MOD(ABS(HASH(n, 'open')), 100) < 5    AS is_open,
       MOD(ABS(HASH(n, 'pd')), 100) < 28                  AS plan_reversed,
       MOD(ABS(HASH(n, 'ire')), 100)                      AS ire_roll,
       MOD(ABS(HASH(n, 'svc')), 5)                        AS svc,
       MOD(ABS(HASH(n, 'wk')), 6)                         AS wk,
       ARRAY_CONSTRUCT('Carmen','José','Luz','Ramón','Ana','Héctor','Gloria','Luis','Milagros','Ángel','Iris','Rafael','Nilda','Pedro','Wanda','Edwin')[MOD(ABS(HASH(n, 'fn')), 16)]::STRING || ' ' ||
       ARRAY_CONSTRUCT('Rivera','Rodríguez','Santiago','Ortiz','Torres','Colón','Vázquez','Pérez','Rosario','Cruz','Figueroa','Meléndez','Negrón','Reyes','Matos','Ramos')[MOD(ABS(HASH(n, 'ln')), 16)]::STRING AS member_name,
       DATEADD(day, -MOD(ABS(HASH(n, 'dob')), 9000), '1958-12-31'::DATE) AS dob,
       TO_VARCHAR(1 + MOD(ABS(HASH(n, 'm1')), 9)) || SUBSTR('ACDEFGHJKMNPQRTUVWXY', 1 + MOD(ABS(HASH(n, 'm2')), 20), 1) || SUBSTR('ACDEFGHJKMNPQRTUVWXY', 1 + MOD(ABS(HASH(n, 'm3')), 20), 1) || TO_VARCHAR(0 + MOD(ABS(HASH(n, 'm4')), 10)) || '-' || SUBSTR('ACDEFGHJKMNPQRTUVWXY', 1 + MOD(ABS(HASH(n, 'm5')), 20), 1) || SUBSTR('ACDEFGHJKMNPQRTUVWXY', 1 + MOD(ABS(HASH(n, 'm6')), 20), 1) || TO_VARCHAR(0 + MOD(ABS(HASH(n, 'm7')), 10)) || '-' || SUBSTR('ACDEFGHJKMNPQRTUVWXY', 1 + MOD(ABS(HASH(n, 'm8')), 20), 1) || SUBSTR('ACDEFGHJKMNPQRTUVWXY', 1 + MOD(ABS(HASH(n, 'm9')), 20), 1) || TO_VARCHAR(0 + MOD(ABS(HASH(n, 'm10')), 10)) || TO_VARCHAR(0 + MOD(ABS(HASH(n, 'm11')), 10)) AS mbi,
       '787-' || TO_VARCHAR(200 + MOD(ABS(HASH(n, 'p1')), 799)) || '-' || LPAD(TO_VARCHAR(MOD(ABS(HASH(n, 'p2')), 10000)), 4, '0') AS phone
FROM b
""",
    """
CREATE OR REPLACE TABLE __FQ__.BRONZE_APPEALS AS
WITH g AS (
    SELECT *,
        CASE
            WHEN n > 300 THEN DATEADD(hour, -(1 + MOD(ABS(HASH(n, 'age')), 40)), CURRENT_TIMESTAMP())
            WHEN is_open THEN DATEADD(hour, -(2 + MOD(ABS(HASH(n, 'age')), 100)), CURRENT_TIMESTAMP())
            ELSE DATEADD(hour, -(24 * 12 + MOD(ABS(HASH(n, 'age')), 24 * 150)), CURRENT_TIMESTAMP())
        END::TIMESTAMP_NTZ AS received_at,
        CASE
            WHEN is_expedited AND is_late THEN 73 + MOD(ABS(HASH(n, 'dh')), 60)
            WHEN is_expedited             THEN  6 + MOD(ABS(HASH(n, 'dh')), 64)
            WHEN is_late                  THEN 24 * (31 + MOD(ABS(HASH(n, 'dh')), 15))
            ELSE                               24 * ( 2 + MOD(ABS(HASH(n, 'dh')), 27))
        END AS decision_hours,
        CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('a lumbar spine MRI','a cardiac catheterization','home health nursing visits','an inpatient rehabilitation stay','a knee arthroscopy')[svc]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('a consultation with an out-of-network oncologist in Miami','an out-of-network cardiothoracic surgeon','follow-up care at a non-contracted hospital in Orlando','an out-of-network pediatric neurologist','a non-contracted dialysis center')[svc]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('a CT scan of the abdomen','an outpatient hernia surgery','a power wheelchair','a sleep study','cataract surgery')[svc]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Ozempic (semaglutide)','Eliquis (apixaban)','Humira (adalimumab)','Jardiance (empagliflozin)','Entresto (sacubitril and valsartan)')[svc]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('proton beam therapy','a genetic tumor profiling test','an investigational stem cell injection','a new gene therapy for macular degeneration','transcranial magnetic stimulation')[svc]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('an emergency room visit','annual wellness visit lab work','an ambulance transport','a colonoscopy billed as diagnostic','a telehealth visit')[svc]::STRING END AS service
    FROM __FQ__._APPEAL_GEN
    WHERE n <= 300
)
SELECT
    'APL-' || LPAD(TO_VARCHAR(n), 6, '0')                                     AS APPEAL_ID,
    'M' || LPAD(TO_VARCHAR(MOD(ABS(HASH(n, 'mid')), 9000000) + 1000000), 8, '0') AS MEMBER_ID,
    member_name                                                               AS MEMBER_NAME,
    mbi                                                                       AS MBI,
    dob                                                                       AS DATE_OF_BIRTH,
    IFF(is_expedited, 'EXPEDITED', 'STANDARD')                                AS APPEAL_TYPE,
    received_at                                                               AS APPEAL_RECEIVED_AT,
    IFF(is_open, NULL, DATEADD(hour, decision_hours, received_at))            AS PLAN_DECISION_AT,
    CASE WHEN is_open THEN 'PENDING'
         WHEN plan_reversed AND NOT is_late THEN 'DENIAL_OVERTURNED_BY_PLAN'
         ELSE 'DENIAL_UPHELD' END                                             AS PLAN_DECISION,
    NOT is_open AND (NOT plan_reversed OR is_late)                            AS FORWARDED_TO_IRE,
    CASE WHEN is_open OR (plan_reversed AND NOT is_late) THEN NULL
         WHEN DATEADD(hour, decision_hours, received_at) > DATEADD(day, -10, CURRENT_TIMESTAMP()) THEN 'PENDING'
         WHEN ire_roll < IFF(is_weak, 62, 14) THEN 'OVERTURNED'
         ELSE 'UPHELD' END                                                    AS IRE_DECISION,
    ARRAY_CONSTRUCT('Medical Necessity','Out-of-Network Access','Prior Authorization','Part D Formulary','Experimental or Investigational','Coding or Billing Error')[IFF(is_mislabeled, MOD(cat + 1 + MOD(n, 4), 6), cat)]::STRING AS HUMAN_CATEGORY,
    REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(
        IFF(is_spanish, CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('El afiliado [[NAME]], fecha de nacimiento [[DOB]], numero de Medicare [[MBI]], apela la denegacion de [[SERVICE]]. El medico indica que el dolor ha empeorado por 6 semanas a pesar de terapia fisica y que el servicio es medicamente necesario. Favor de llamar al [[PHONE]].')[0]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('Escribo por mi familiar [[NAME]], fecha de nacimiento [[DOB]], MBI [[MBI]]. El plan denego [[SERVICE]]. No hay especialista en la red de Triple-S en Puerto Rico que haga este procedimiento y el medico lo refirio fuera de la red. Mi telefono es [[PHONE]].')[0]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('Apelacion de [[NAME]], fecha de nacimiento [[DOB]], numero de Medicare [[MBI]]. Denegaron [[SERVICE]] por falta de preautorizacion, pero la oficina del medico dice que enviaron la solicitud por fax antes de la cita. Contacto [[PHONE]].')[0]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Afiliado(a) [[NAME]] (fecha de nacimiento [[DOB]], MBI [[MBI]]) solicita cobertura de [[SERVICE]]. El plan dice que no esta en el formulario. Ya probo otras dos alternativas sin resultado y el medico dice que es la unica opcion. Llamar al [[PHONE]].')[0]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('[[NAME]], fecha de nacimiento [[DOB]], MBI [[MBI]], apela la denegacion de [[SERVICE]]. El oncologo lo recomienda porque el tratamiento estandar fallo. Favor de contactar al [[PHONE]].')[0]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('Afiliado(a) [[NAME]] (fecha de nacimiento [[DOB]], MBI [[MBI]]) no esta de acuerdo con la denegacion de [[SERVICE]]. Cree que el proveedor uso el codigo de facturacion incorrecto y que debio cubrirse como servicio preventivo. Telefono [[PHONE]].')[0]::STRING END, CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('Member [[NAME]] (DOB [[DOB]], Medicare ID [[MBI]]) is appealing the denial of [[SERVICE]]. The treating physician states the pain has worsened for 6 weeks despite physical therapy and medication and that the service is medically necessary to avoid a hospital admission. Please call the member at [[PHONE]].')[0]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('I am writing on behalf of my family member [[NAME]], born [[DOB]], MBI [[MBI]]. The plan denied [[SERVICE]]. There is no specialist in the Triple-S network in Puerto Rico who performs this procedure, and the doctor made a referral outside the network. You can reach me at [[PHONE]].')[0]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('Appeal for [[NAME]], DOB [[DOB]], Medicare number [[MBI]]. The claim for [[SERVICE]] was denied because there was no prior authorization. The doctor office says they submitted the authorization request by fax before the service date. Contact [[PHONE]].')[0]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Member [[NAME]] (DOB [[DOB]], MBI [[MBI]]) requests coverage of [[SERVICE]]. The plan says the drug is not on the formulary. The member already tried metformin and two other alternatives without success and the doctor says this medication is the only option. Call [[PHONE]].')[0]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('[[NAME]], date of birth [[DOB]], MBI [[MBI]], is appealing the denial of [[SERVICE]]. The oncologist recommends it because standard treatment failed, and the member read that it is being used at major cancer centers. Please contact [[PHONE]].')[0]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('Member [[NAME]] (DOB [[DOB]], MBI [[MBI]]) disputes the denial for [[SERVICE]]. The member believes the provider used the wrong billing code and the service should have been covered as a preventive benefit. Member phone [[PHONE]].')[0]::STRING END),
        '[[NAME]]', member_name), '[[DOB]]', TO_VARCHAR(dob, 'MM/DD/YYYY')), '[[MBI]]', mbi),
        '[[PHONE]]', phone), '[[SERVICE]]', service)                          AS APPEAL_REASON_TEXT,
    IFF(is_weak, ARRAY_CONSTRUCT('Request denied. Not covered.','Denied per plan policy.','Does not meet criteria.','Service not medically necessary.','Denied - see Evidence of Coverage.','Not approved at this time.')[wk]::STRING,
        REPLACE(CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('Denied: InterQual criteria for [[SERVICE]] not met. Records from the last 30 days show no documented trial of conservative therapy; the requested 6 weeks of physical therapy notes were not received by the review deadline.')[0]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('Denied: in-network specialists with equivalent expertise are available within 30 miles at Centro Medico in San Juan. No continuity-of-care or network adequacy exception criteria under Evidence of Coverage chapter 4 were documented.')[0]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('Denied: no prior authorization request for [[SERVICE]] was received before the date of service. Policy UM-102 requires prior authorization; fax logs show no submission and retro-authorization criteria were not met.')[0]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Denied: requested drug is non-formulary for the plan year. Records show only one formulary alternative was tried; the prescriber did not submit a formulary exception with clinical justification as required by Part D rules.')[0]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('Denied: [[SERVICE]] is considered investigational for this diagnosis under the applicable CMS National Coverage Determination and plan medical policy MP-220. No peer-reviewed evidence supporting this indication was submitted.')[0]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('Denied: the claim was submitted with a CPT code that does not match the documented service in the medical record. The provider was notified on file to submit a corrected claim.')[0]::STRING END, '[[SERVICE]]', service))              AS PLAN_RATIONALE_TEXT
FROM g
""",
    """
CREATE OR REPLACE TABLE __FQ__.BRONZE_APPEALS_NEW_BATCH AS
WITH g AS (
    SELECT *,
        CASE
            WHEN n > 300 THEN DATEADD(hour, -(1 + MOD(ABS(HASH(n, 'age')), 40)), CURRENT_TIMESTAMP())
            WHEN is_open THEN DATEADD(hour, -(2 + MOD(ABS(HASH(n, 'age')), 100)), CURRENT_TIMESTAMP())
            ELSE DATEADD(hour, -(24 * 12 + MOD(ABS(HASH(n, 'age')), 24 * 150)), CURRENT_TIMESTAMP())
        END::TIMESTAMP_NTZ AS received_at,
        CASE
            WHEN is_expedited AND is_late THEN 73 + MOD(ABS(HASH(n, 'dh')), 60)
            WHEN is_expedited             THEN  6 + MOD(ABS(HASH(n, 'dh')), 64)
            WHEN is_late                  THEN 24 * (31 + MOD(ABS(HASH(n, 'dh')), 15))
            ELSE                               24 * ( 2 + MOD(ABS(HASH(n, 'dh')), 27))
        END AS decision_hours,
        CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('a lumbar spine MRI','a cardiac catheterization','home health nursing visits','an inpatient rehabilitation stay','a knee arthroscopy')[svc]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('a consultation with an out-of-network oncologist in Miami','an out-of-network cardiothoracic surgeon','follow-up care at a non-contracted hospital in Orlando','an out-of-network pediatric neurologist','a non-contracted dialysis center')[svc]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('a CT scan of the abdomen','an outpatient hernia surgery','a power wheelchair','a sleep study','cataract surgery')[svc]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Ozempic (semaglutide)','Eliquis (apixaban)','Humira (adalimumab)','Jardiance (empagliflozin)','Entresto (sacubitril and valsartan)')[svc]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('proton beam therapy','a genetic tumor profiling test','an investigational stem cell injection','a new gene therapy for macular degeneration','transcranial magnetic stimulation')[svc]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('an emergency room visit','annual wellness visit lab work','an ambulance transport','a colonoscopy billed as diagnostic','a telehealth visit')[svc]::STRING END AS service
    FROM __FQ__._APPEAL_GEN
    WHERE n > 300
)
SELECT
    'APL-' || LPAD(TO_VARCHAR(n), 6, '0')                                     AS APPEAL_ID,
    'M' || LPAD(TO_VARCHAR(MOD(ABS(HASH(n, 'mid')), 9000000) + 1000000), 8, '0') AS MEMBER_ID,
    member_name                                                               AS MEMBER_NAME,
    mbi                                                                       AS MBI,
    dob                                                                       AS DATE_OF_BIRTH,
    IFF(is_expedited, 'EXPEDITED', 'STANDARD')                                AS APPEAL_TYPE,
    received_at                                                               AS APPEAL_RECEIVED_AT,
    IFF(is_open, NULL, DATEADD(hour, decision_hours, received_at))            AS PLAN_DECISION_AT,
    CASE WHEN is_open THEN 'PENDING'
         WHEN plan_reversed AND NOT is_late THEN 'DENIAL_OVERTURNED_BY_PLAN'
         ELSE 'DENIAL_UPHELD' END                                             AS PLAN_DECISION,
    NOT is_open AND (NOT plan_reversed OR is_late)                            AS FORWARDED_TO_IRE,
    CASE WHEN is_open OR (plan_reversed AND NOT is_late) THEN NULL
         WHEN DATEADD(hour, decision_hours, received_at) > DATEADD(day, -10, CURRENT_TIMESTAMP()) THEN 'PENDING'
         WHEN ire_roll < IFF(is_weak, 62, 14) THEN 'OVERTURNED'
         ELSE 'UPHELD' END                                                    AS IRE_DECISION,
    ARRAY_CONSTRUCT('Medical Necessity','Out-of-Network Access','Prior Authorization','Part D Formulary','Experimental or Investigational','Coding or Billing Error')[IFF(is_mislabeled, MOD(cat + 1 + MOD(n, 4), 6), cat)]::STRING AS HUMAN_CATEGORY,
    REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(
        IFF(is_spanish, CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('El afiliado [[NAME]], fecha de nacimiento [[DOB]], numero de Medicare [[MBI]], apela la denegacion de [[SERVICE]]. El medico indica que el dolor ha empeorado por 6 semanas a pesar de terapia fisica y que el servicio es medicamente necesario. Favor de llamar al [[PHONE]].')[0]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('Escribo por mi familiar [[NAME]], fecha de nacimiento [[DOB]], MBI [[MBI]]. El plan denego [[SERVICE]]. No hay especialista en la red de Triple-S en Puerto Rico que haga este procedimiento y el medico lo refirio fuera de la red. Mi telefono es [[PHONE]].')[0]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('Apelacion de [[NAME]], fecha de nacimiento [[DOB]], numero de Medicare [[MBI]]. Denegaron [[SERVICE]] por falta de preautorizacion, pero la oficina del medico dice que enviaron la solicitud por fax antes de la cita. Contacto [[PHONE]].')[0]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Afiliado(a) [[NAME]] (fecha de nacimiento [[DOB]], MBI [[MBI]]) solicita cobertura de [[SERVICE]]. El plan dice que no esta en el formulario. Ya probo otras dos alternativas sin resultado y el medico dice que es la unica opcion. Llamar al [[PHONE]].')[0]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('[[NAME]], fecha de nacimiento [[DOB]], MBI [[MBI]], apela la denegacion de [[SERVICE]]. El oncologo lo recomienda porque el tratamiento estandar fallo. Favor de contactar al [[PHONE]].')[0]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('Afiliado(a) [[NAME]] (fecha de nacimiento [[DOB]], MBI [[MBI]]) no esta de acuerdo con la denegacion de [[SERVICE]]. Cree que el proveedor uso el codigo de facturacion incorrecto y que debio cubrirse como servicio preventivo. Telefono [[PHONE]].')[0]::STRING END, CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('Member [[NAME]] (DOB [[DOB]], Medicare ID [[MBI]]) is appealing the denial of [[SERVICE]]. The treating physician states the pain has worsened for 6 weeks despite physical therapy and medication and that the service is medically necessary to avoid a hospital admission. Please call the member at [[PHONE]].')[0]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('I am writing on behalf of my family member [[NAME]], born [[DOB]], MBI [[MBI]]. The plan denied [[SERVICE]]. There is no specialist in the Triple-S network in Puerto Rico who performs this procedure, and the doctor made a referral outside the network. You can reach me at [[PHONE]].')[0]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('Appeal for [[NAME]], DOB [[DOB]], Medicare number [[MBI]]. The claim for [[SERVICE]] was denied because there was no prior authorization. The doctor office says they submitted the authorization request by fax before the service date. Contact [[PHONE]].')[0]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Member [[NAME]] (DOB [[DOB]], MBI [[MBI]]) requests coverage of [[SERVICE]]. The plan says the drug is not on the formulary. The member already tried metformin and two other alternatives without success and the doctor says this medication is the only option. Call [[PHONE]].')[0]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('[[NAME]], date of birth [[DOB]], MBI [[MBI]], is appealing the denial of [[SERVICE]]. The oncologist recommends it because standard treatment failed, and the member read that it is being used at major cancer centers. Please contact [[PHONE]].')[0]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('Member [[NAME]] (DOB [[DOB]], MBI [[MBI]]) disputes the denial for [[SERVICE]]. The member believes the provider used the wrong billing code and the service should have been covered as a preventive benefit. Member phone [[PHONE]].')[0]::STRING END),
        '[[NAME]]', member_name), '[[DOB]]', TO_VARCHAR(dob, 'MM/DD/YYYY')), '[[MBI]]', mbi),
        '[[PHONE]]', phone), '[[SERVICE]]', service)                          AS APPEAL_REASON_TEXT,
    IFF(is_weak, ARRAY_CONSTRUCT('Request denied. Not covered.','Denied per plan policy.','Does not meet criteria.','Service not medically necessary.','Denied - see Evidence of Coverage.','Not approved at this time.')[wk]::STRING,
        REPLACE(CASE cat WHEN 0 THEN ARRAY_CONSTRUCT('Denied: InterQual criteria for [[SERVICE]] not met. Records from the last 30 days show no documented trial of conservative therapy; the requested 6 weeks of physical therapy notes were not received by the review deadline.')[0]::STRING WHEN 1 THEN ARRAY_CONSTRUCT('Denied: in-network specialists with equivalent expertise are available within 30 miles at Centro Medico in San Juan. No continuity-of-care or network adequacy exception criteria under Evidence of Coverage chapter 4 were documented.')[0]::STRING WHEN 2 THEN ARRAY_CONSTRUCT('Denied: no prior authorization request for [[SERVICE]] was received before the date of service. Policy UM-102 requires prior authorization; fax logs show no submission and retro-authorization criteria were not met.')[0]::STRING WHEN 3 THEN ARRAY_CONSTRUCT('Denied: requested drug is non-formulary for the plan year. Records show only one formulary alternative was tried; the prescriber did not submit a formulary exception with clinical justification as required by Part D rules.')[0]::STRING WHEN 4 THEN ARRAY_CONSTRUCT('Denied: [[SERVICE]] is considered investigational for this diagnosis under the applicable CMS National Coverage Determination and plan medical policy MP-220. No peer-reviewed evidence supporting this indication was submitted.')[0]::STRING WHEN 5 THEN ARRAY_CONSTRUCT('Denied: the claim was submitted with a CPT code that does not match the documented service in the medical record. The provider was notified on file to submit a corrected claim.')[0]::STRING END, '[[SERVICE]]', service))              AS PLAN_RATIONALE_TEXT
FROM g
""",
    """
ALTER TABLE __FQ__.BRONZE_APPEALS SET CHANGE_TRACKING = TRUE
""",
    """
DROP TABLE IF EXISTS __FQ__._APPEAL_GEN
"""
]

for i, stmt in enumerate(LOAD_STATEMENTS, 1):
    session.sql(stmt.replace("__FQ__", fq)).collect()
    print(f"Step {i}/{len(LOAD_STATEMENTS)} done")
print("BRONZE layer loaded.")

In [ ]:
%%sql -r bronze_counts
SELECT 'BRONZE_APPEALS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM {{fq}}.BRONZE_APPEALS
UNION ALL
SELECT 'BRONZE_APPEALS_NEW_BATCH', COUNT(*) FROM {{fq}}.BRONZE_APPEALS_NEW_BATCH

---
# Section 1 — Explore BRONZE and estimate AI cost (7 min)

> **Cortex Code prompt**
> ```
> Explain the table BRONZE_APPEALS in my schema TSS_SNOWDAY.LAB2_<MY_USER>.
> Which columns contain PHI, either structured or inside free text? Show me 3 sample
> rows, and summarize appeals by APPEAL_TYPE, PLAN_DECISION and IRE_DECISION.
> ```

In [ ]:
%%sql -r bronze_sample
SELECT APPEAL_ID, MEMBER_NAME, MBI, DATE_OF_BIRTH, APPEAL_TYPE, PLAN_DECISION, IRE_DECISION,
       HUMAN_CATEGORY, APPEAL_REASON_TEXT, PLAN_RATIONALE_TEXT
FROM {{fq}}.BRONZE_APPEALS
ORDER BY APPEAL_ID
LIMIT 3

In [ ]:
%%sql -r bronze_profile
SELECT APPEAL_TYPE, PLAN_DECISION, IRE_DECISION, COUNT(*) AS APPEALS
FROM {{fq}}.BRONZE_APPEALS
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

### Ask before you spend
AI functions are billed by **tokens**. A productive engineer estimates cost **before** running AI on a full table.

> **Cortex Code prompt**
> ```
> Before I run AI_REDACT and AI_CLASSIFY on APPEAL_REASON_TEXT, estimate how many
> tokens that would consume for the whole table using AI_COUNT_TOKENS.
> ```

In [ ]:
%%sql -r token_estimate
SELECT
    COUNT(*)                                                      AS APPEALS,
    SUM(AI_COUNT_TOKENS('ai_redact', APPEAL_REASON_TEXT))         AS AI_REDACT_INPUT_TOKENS,
    ROUND(AVG(AI_COUNT_TOKENS('ai_redact', APPEAL_REASON_TEXT)))  AS AVG_TOKENS_PER_APPEAL
FROM {{fq}}.BRONZE_APPEALS

Keep `AVG_TOKENS_PER_APPEAL` in mind. In Section 5 we will see that, with incremental Dynamic Tables, you pay that cost **once per appeal**, not once per refresh.

---
# Section 2 — Protect PHI (15 min)

PHI shows up in **two shapes**, and each needs a different tool:

| PHI shape | Example | Snowflake tool |
|---|---|---|
| **Structured** columns | `MEMBER_NAME`, `MBI`, `DATE_OF_BIRTH` | **Masking policies**: governed, role-based, no data copy |
| **Unstructured** free text | *"Member Carmen Rivera (DOB 03/12/1950, MBI 1EG4-TE5-MK73)..."* | **AI_REDACT**: an LLM finds and replaces PII inside text |

## 2.1 Masking policies for structured PHI

> **Cortex Code prompt**
> ```
> Create masking policies so that MEMBER_NAME and MBI in BRONZE_APPEALS are only
> visible to the role TSS_PHI_READER, and DATE_OF_BIRTH shows only the year for
> everyone else. Apply them and show me what my role sees.
> ```

In [ ]:
%%sql -r mask_string_policy
CREATE MASKING POLICY IF NOT EXISTS {{fq}}.PHI_STRING_MASK AS (val STRING) RETURNS STRING ->
    CASE WHEN IS_ROLE_IN_SESSION('TSS_PHI_READER') THEN val ELSE '*** PHI MASKED ***' END
    COMMENT = 'Triple-S Snow Day: only TSS_PHI_READER sees structured PHI'

In [ ]:
%%sql -r mask_date_policy
CREATE MASKING POLICY IF NOT EXISTS {{fq}}.PHI_DATE_MASK AS (val DATE) RETURNS DATE ->
    CASE WHEN IS_ROLE_IN_SESSION('TSS_PHI_READER') THEN val ELSE DATE_FROM_PARTS(YEAR(val), 1, 1) END
    COMMENT = 'Triple-S Snow Day: date of birth reduced to year for non-PHI roles'

In [ ]:
%%sql -r apply_mask_name
ALTER TABLE {{fq}}.BRONZE_APPEALS MODIFY COLUMN MEMBER_NAME SET MASKING POLICY {{fq}}.PHI_STRING_MASK FORCE

In [ ]:
%%sql -r apply_mask_mbi
ALTER TABLE {{fq}}.BRONZE_APPEALS MODIFY COLUMN MBI SET MASKING POLICY {{fq}}.PHI_STRING_MASK FORCE

In [ ]:
%%sql -r apply_mask_dob
ALTER TABLE {{fq}}.BRONZE_APPEALS MODIFY COLUMN DATE_OF_BIRTH SET MASKING POLICY {{fq}}.PHI_DATE_MASK FORCE

In [ ]:
%%sql -r after_masking
SELECT APPEAL_ID, MEMBER_NAME, MBI, DATE_OF_BIRTH, APPEAL_REASON_TEXT
FROM {{fq}}.BRONZE_APPEALS
ORDER BY APPEAL_ID
LIMIT 3

**The structured columns are protected, but look at `APPEAL_REASON_TEXT`.** The member's name, date of birth, MBI and phone number are all still there in the free text. A masking policy can't fix that without hiding the whole column, and we *need* the text to understand why members appeal.

## 2.2 Prototype AI_REDACT on 5 rows first

> **Cortex Code prompt** (habit: prototype small, then validate)
> ```
> Test AI_REDACT on 5 appeals from BRONZE_APPEALS, including at least 2 written
> in Spanish. Show the original text next to the redacted text.
> ```

In [ ]:
%%sql -r redact_prototype
WITH sample_rows AS (
    (SELECT APPEAL_ID, APPEAL_REASON_TEXT FROM {{fq}}.BRONZE_APPEALS
      WHERE APPEAL_REASON_TEXT ILIKE '%fecha de nacimiento%'
      ORDER BY APPEAL_ID LIMIT 2)
    UNION ALL
    (SELECT APPEAL_ID, APPEAL_REASON_TEXT FROM {{fq}}.BRONZE_APPEALS
      WHERE APPEAL_REASON_TEXT ILIKE '%DOB%' ORDER BY APPEAL_ID LIMIT 3)
)
SELECT APPEAL_ID,
       APPEAL_REASON_TEXT            AS ORIGINAL_TEXT,
       AI_REDACT(APPEAL_REASON_TEXT)::STRING AS REDACTED_TEXT
FROM sample_rows

Read the output carefully, **field by field**. Did *every* piece of PHI get replaced: name, date of birth, phone **and the Medicare ID (MBI)**? Did anything that is *not* PHI get redacted by mistake?

AI_REDACT is a best-effort AI model. Snowflake's documentation says to **always review the output**. Never assume an AI step worked: **measure it.** We will do that across every row as soon as the SILVER layer exists.

## 2.3 Productionize: the SILVER Dynamic Table

Now that the prototype looks reasonable, let Cortex Code build the production object.

> **Cortex Code prompt** (habit: plan first)
> ```
> Create a table SILVER_APPEALS from BRONZE_APPEALS. Drop MEMBER_NAME, MBI and DATE_OF_BIRTH entirely (data minimization),
> keep the operational columns, redact APPEAL_REASON_TEXT with AI_REDACT, and add
> a PHI_LEAK_SUSPECTED flag that uses regex to detect any MBI, phone number or date
> that survived redaction. Explain your design before creating it.
> ```

> **Design note: why SILVER is a table and not a Dynamic Table.** When we built this lab, AI_REDACT worked on the *initial* build of an incremental Dynamic Table but **failed on later incremental refreshes**, while AI_CLASSIFY, AI_FILTER and AI_EXTRACT refreshed incrementally without problems. So SILVER is a regular table, and in Section 5 we load it **incrementally ourselves** by redacting only new appeals. Testing every function in its real refresh mode **before** go-live is part of the job.

*This cell runs AI_REDACT on 300 appeals. It takes about 1 to 2 minutes.*

In [ ]:
%%sql -r create_silver
CREATE OR REPLACE TABLE {{fq}}.SILVER_APPEALS
    CHANGE_TRACKING = TRUE
    COMMENT = 'Appeals with structured PHI removed and free text redacted by AI_REDACT'
AS
SELECT
    APPEAL_ID, MEMBER_ID, APPEAL_TYPE, APPEAL_RECEIVED_AT, PLAN_DECISION_AT, PLAN_DECISION,
    FORWARDED_TO_IRE, IRE_DECISION, HUMAN_CATEGORY, PLAN_RATIONALE_TEXT,
    redacted AS APPEAL_REASON_REDACTED,
    REGEXP_LIKE(redacted, '.*[1-9][A-Z]{2}[0-9]-?[A-Z]{2}[0-9]-?[A-Z]{2}[0-9]{2}.*')
      OR REGEXP_LIKE(redacted, '.*787-[0-9]{3}-[0-9]{4}.*')
      OR REGEXP_LIKE(redacted, '.*[0-9]{2}/[0-9]{2}/[0-9]{4}.*') AS PHI_LEAK_SUSPECTED
FROM (SELECT *, AI_REDACT(APPEAL_REASON_TEXT)::STRING AS redacted FROM {{fq}}.BRONZE_APPEALS)

### Validate the redaction across every row

In [ ]:
%%sql -r redaction_leak_report
SELECT
    COUNT(*)                                                                                   AS APPEALS,
    COUNT_IF(REGEXP_LIKE(APPEAL_REASON_REDACTED, '.*[1-9][A-Z]{2}[0-9]-?[A-Z]{2}[0-9]-?[A-Z]{2}[0-9]{2}.*')) AS MBI_LEAKED,
    COUNT_IF(REGEXP_LIKE(APPEAL_REASON_REDACTED, '.*787-[0-9]{3}-[0-9]{4}.*'))                  AS PHONE_LEAKED,
    COUNT_IF(REGEXP_LIKE(APPEAL_REASON_REDACTED, '.*[0-9]{2}/[0-9]{2}/[0-9]{4}.*'))             AS DOB_LEAKED,
    COUNT_IF(APPEAL_REASON_REDACTED ILIKE '%[ADDRESS]%')                                       AS ADDRESS_PLACEHOLDERS,
    ROUND(100 * COUNT_IF(PHI_LEAK_SUSPECTED) / COUNT(*), 1)                                    AS PCT_ROWS_WITH_LEAK
FROM {{fq}}.SILVER_APPEALS

In [ ]:
%%sql -r redaction_leaks
SELECT APPEAL_ID, APPEAL_REASON_REDACTED
FROM {{fq}}.SILVER_APPEALS
WHERE PHI_LEAK_SUSPECTED
LIMIT 10

### What the validation found
Names, dates of birth and phone numbers are redacted (in English **and** Spanish), but **the Medicare Beneficiary Identifier (MBI) survives in almost every row**. The MBI is not one of the PII categories AI_REDACT detects. You can also see a false positive: *"Puerto Rico"* sometimes becomes `[ADDRESS]`.

**Without this check, you would have published Medicare IDs to every analyst who can read SILVER.** This is the most important habit in the lab: **an AI step is not done until you have measured it.**

## 2.4 Fix it: add a deterministic safety net

The fix is not a better prompt. It is **defense in depth**: AI_REDACT for free-form PII, plus a regex pass for identifiers with a known format (MBI, phone, dates). Regex functions are deterministic and cost nothing in AI tokens.

> **Cortex Code prompt**
> ```
> AI_REDACT is not catching the MBI. Update SILVER_APPEALS so that after AI_REDACT
> a REGEXP_REPLACE pass replaces any MBI pattern with [MBI], any 787 phone with
> [PHONE_NUMBER] and any MM/DD/YYYY date with [DATE]. Then re-run the leak report.
> ```

In [ ]:
%%sql -r create_silver_v2
CREATE OR REPLACE TABLE {{fq}}.SILVER_APPEALS
    CHANGE_TRACKING = TRUE
    COMMENT = 'Appeals with structured PHI removed; free text redacted by AI_REDACT plus a regex safety net'
AS
SELECT
    APPEAL_ID, MEMBER_ID, APPEAL_TYPE, APPEAL_RECEIVED_AT, PLAN_DECISION_AT, PLAN_DECISION,
    FORWARDED_TO_IRE, IRE_DECISION, HUMAN_CATEGORY, PLAN_RATIONALE_TEXT,
    redacted AS APPEAL_REASON_REDACTED,
    REGEXP_LIKE(redacted, '.*[1-9][A-Z]{2}[0-9]-?[A-Z]{2}[0-9]-?[A-Z]{2}[0-9]{2}.*')
      OR REGEXP_LIKE(redacted, '.*787-[0-9]{3}-[0-9]{4}.*')
      OR REGEXP_LIKE(redacted, '.*[0-9]{2}/[0-9]{2}/[0-9]{4}.*') AS PHI_LEAK_SUSPECTED
FROM (
    SELECT *,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(
            AI_REDACT(APPEAL_REASON_TEXT)::STRING,
            '[1-9][A-Z]{2}[0-9]-?[A-Z]{2}[0-9]-?[A-Z]{2}[0-9]{2}', '[MBI]'),
            '787-[0-9]{3}-[0-9]{4}', '[PHONE_NUMBER]'),
            '[0-9]{2}/[0-9]{2}/[0-9]{4}', '[DATE]') AS redacted
    FROM {{fq}}.BRONZE_APPEALS
)

In [ ]:
%%sql -r redaction_leak_report_v2
SELECT
    COUNT(*)                                                                                   AS APPEALS,
    COUNT_IF(REGEXP_LIKE(APPEAL_REASON_REDACTED, '.*[1-9][A-Z]{2}[0-9]-?[A-Z]{2}[0-9]-?[A-Z]{2}[0-9]{2}.*')) AS MBI_LEAKED,
    COUNT_IF(REGEXP_LIKE(APPEAL_REASON_REDACTED, '.*787-[0-9]{3}-[0-9]{4}.*'))                  AS PHONE_LEAKED,
    COUNT_IF(REGEXP_LIKE(APPEAL_REASON_REDACTED, '.*[0-9]{2}/[0-9]{2}/[0-9]{4}.*'))             AS DOB_LEAKED,
    COUNT_IF(APPEAL_REASON_REDACTED ILIKE '%[ADDRESS]%')                                       AS ADDRESS_PLACEHOLDERS,
    ROUND(100 * COUNT_IF(PHI_LEAK_SUSPECTED) / COUNT(*), 1)                                    AS PCT_ROWS_WITH_LEAK
FROM {{fq}}.SILVER_APPEALS

Every leak counter should now be **0**. You have a SILVER layer analysts can safely use.

**Discuss:** in production, what else would you add?
* **Quarantine** any row where `PHI_LEAK_SUSPECTED = TRUE` and route it to a human reviewer
* Run a weekly **sample audit** of SILVER text by the Privacy Office
* Decide with Compliance whether false positives like *Puerto Rico → [ADDRESS]* are acceptable

---
# Section 3 — Classify and enrich with AI (15 min)

We now want three new fields on every appeal, computed from the **redacted** text:

| Field | AI function | Why the business cares |
|---|---|---|
| `AI_CATEGORY` | **AI_CLASSIFY** | The appeal reason, consistent and not typed by hand |
| `IS_WEAK_RATIONALE` | **AI_FILTER** | Vague denial reasons are the ones the IRE tends to overturn |
| `REQUESTED_SERVICE`, `MEMBER_ARGUMENT` | **AI_EXTRACT** | Structured fields pulled from free text, for reporting |

## 3.1 Prototype on 10 rows

> **Cortex Code prompt**
> ```
> On 10 rows of SILVER_APPEALS, try:
> - AI_CLASSIFY on APPEAL_REASON_REDACTED with these labels and short descriptions:
>   Medical Necessity, Out-of-Network Access, Prior Authorization, Part D Formulary,
>   Experimental or Investigational, Coding or Billing Error
> - AI_FILTER on PLAN_RATIONALE_TEXT: is the denial rationale vague, without citing a
>   specific criterion, policy, missing document or alternative?
> - AI_EXTRACT the requested service and the member's main argument
> Show the results next to HUMAN_CATEGORY.
> ```

In [ ]:
%%sql -r enrich_prototype
SELECT
    APPEAL_ID,
    HUMAN_CATEGORY,
    AI_CLASSIFY(
        APPEAL_REASON_REDACTED,
        [
            {'label': 'Medical Necessity',               'description': 'member or doctor argues the service is clinically needed'},
            {'label': 'Out-of-Network Access',           'description': 'member wants a provider or facility outside the plan network'},
            {'label': 'Prior Authorization',             'description': 'denied because authorization was missing or not received in time'},
            {'label': 'Part D Formulary',                'description': 'prescription drug not on the formulary or step therapy'},
            {'label': 'Experimental or Investigational', 'description': 'new, unproven or investigational treatment'},
            {'label': 'Coding or Billing Error',         'description': 'wrong billing or procedure code, or claim processing dispute'}
        ],
        {'task_description': 'Classify why a Medicare Advantage member is appealing a denial'}
    ):labels[0]::STRING                                                         AS AI_CATEGORY,
    PLAN_RATIONALE_TEXT,
    AI_FILTER(PROMPT(
        'This health plan denial rationale is vague or generic: it does NOT cite a specific clinical criterion, policy number, missing document or covered alternative. Rationale: {0}',
        PLAN_RATIONALE_TEXT))                                                   AS IS_WEAK_RATIONALE,
    AI_EXTRACT(
        text => APPEAL_REASON_REDACTED,
        responseFormat => {'requested_service': 'What service, procedure, drug or equipment is being requested?',
                            'member_argument':   'In one short sentence, why does the member or doctor say it should be covered?'}
    ):response                                                                  AS EXTRACTED
FROM {{fq}}.SILVER_APPEALS
ORDER BY APPEAL_ID
LIMIT 10

**Iterate like an engineer:** if a label looks wrong, don't accept it. Ask Cortex Code *"Why was APL-0000xx classified as Prior Authorization? Improve the label descriptions and re-test on the same 10 rows."* Small label descriptions often change accuracy a lot.

## 3.2 Productionize: the ENRICHED Dynamic Table
> **Cortex Code prompt**
> *"Turn the prototype into an incremental Dynamic Table ENRICHED_APPEALS on top of the SILVER_APPEALS table. Keep the AI functions in the SELECT clause so it stays incremental, and flatten the AI_EXTRACT output into columns."*

*Runs three AI functions on 300 appeals. It takes about 2 to 3 minutes.*

In [ ]:
%%sql -r create_enriched
CREATE OR REPLACE DYNAMIC TABLE {{fq}}.ENRICHED_APPEALS
    TARGET_LAG = '1 day'
    WAREHOUSE = {{wh}}
    REFRESH_MODE = INCREMENTAL
    COMMENT = 'Appeals enriched with AI_CLASSIFY, AI_FILTER and AI_EXTRACT on redacted text'
AS
SELECT
    s.*,
    AI_CLASSIFY(
        APPEAL_REASON_REDACTED,
        [
            {'label': 'Medical Necessity',               'description': 'member or doctor argues the service is clinically needed'},
            {'label': 'Out-of-Network Access',           'description': 'member wants a provider or facility outside the plan network'},
            {'label': 'Prior Authorization',             'description': 'denied because authorization was missing or not received in time'},
            {'label': 'Part D Formulary',                'description': 'prescription drug not on the formulary or step therapy'},
            {'label': 'Experimental or Investigational', 'description': 'new, unproven or investigational treatment'},
            {'label': 'Coding or Billing Error',         'description': 'wrong billing or procedure code, or claim processing dispute'}
        ],
        {'task_description': 'Classify why a Medicare Advantage member is appealing a denial'}
    ):labels[0]::STRING AS AI_CATEGORY,
    AI_FILTER(PROMPT(
        'This health plan denial rationale is vague or generic: it does NOT cite a specific clinical criterion, policy number, missing document or covered alternative. Rationale: {0}',
        PLAN_RATIONALE_TEXT)) AS IS_WEAK_RATIONALE,
    AI_EXTRACT(
        text => APPEAL_REASON_REDACTED,
        responseFormat => {'requested_service': 'What service, procedure, drug or equipment is being requested?',
                            'member_argument':   'In one short sentence, why does the member or doctor say it should be covered?'}
    ) AS EXTRACTED_RAW,
    EXTRACTED_RAW:response:requested_service::STRING AS REQUESTED_SERVICE,
    EXTRACTED_RAW:response:member_argument::STRING   AS MEMBER_ARGUMENT
FROM {{fq}}.SILVER_APPEALS s

In [ ]:
%%sql -r enriched_sample
SELECT APPEAL_ID, HUMAN_CATEGORY, AI_CATEGORY, IS_WEAK_RATIONALE, REQUESTED_SERVICE, MEMBER_ARGUMENT
FROM {{fq}}.ENRICHED_APPEALS
ORDER BY APPEAL_ID
LIMIT 10

---
# Section 4 — Validate AI against the humans (10 min)

Your appeals team has been typing `HUMAN_CATEGORY` by hand for years. Who is right when the AI and the human disagree?

> **Cortex Code prompt**
> ```
> Compare AI_CATEGORY with HUMAN_CATEGORY in ENRICHED_APPEALS. Give me the agreement
> rate, a confusion matrix, and 10 examples where they disagree with the redacted
> text. For each example, tell me which label you think is correct and why.
> ```

In [ ]:
%%sql -r agreement_rate
SELECT
    COUNT(*)                                        AS APPEALS,
    COUNT_IF(AI_CATEGORY = HUMAN_CATEGORY)          AS AGREE,
    COUNT_IF(AI_CATEGORY <> HUMAN_CATEGORY)         AS DISAGREE,
    ROUND(100 * AGREE / APPEALS, 1)                 AS AGREEMENT_PCT
FROM {{fq}}.ENRICHED_APPEALS

In [ ]:
%%sql -r confusion_matrix
SELECT HUMAN_CATEGORY, AI_CATEGORY, COUNT(*) AS APPEALS
FROM {{fq}}.ENRICHED_APPEALS
WHERE AI_CATEGORY <> HUMAN_CATEGORY
GROUP BY 1, 2
ORDER BY 3 DESC

In [ ]:
%%sql -r disagreements
SELECT APPEAL_ID, HUMAN_CATEGORY, AI_CATEGORY, APPEAL_REASON_REDACTED
FROM {{fq}}.ENRICHED_APPEALS
WHERE AI_CATEGORY <> HUMAN_CATEGORY
ORDER BY APPEAL_ID
LIMIT 10

### What to look for
Read 3 or 4 disagreements yourself. In our test run agreement was **about 79%**, and the disagreements came in **two kinds**:

1. **Human labeling errors.** For example, the text clearly describes a non-formulary drug (*Humira*) but the human typed *Experimental or Investigational*. Reports built on `HUMAN_CATEGORY` today are **silently wrong** for these appeals.
2. **AI confusion between similar labels.** The largest cell in the confusion matrix is usually human *Experimental or Investigational* vs. AI *Medical Necessity*. The text says *"standard treatment failed"*, which sounds like a medical-necessity argument, but the service (proton beam therapy, gene therapy) is investigational.

Neither the human nor the AI is "the truth". **Validation is a practice, not a one-time step.**

## 4.1 Iterate with Cortex Code (optional, 5 min)
> **Cortex Code prompt**
> ```
> Most disagreements are human 'Experimental or Investigational' vs AI 'Medical Necessity'.
> Improve the label descriptions so a request for a new, unproven or investigational
> service is classified as Experimental even if the member argues it is necessary.
> Re-test on the disagreeing rows only and show me the new agreement rate.
> ```

> **Then:** *"Create a table APPEAL_LABEL_REVIEW with the remaining disagreements so a nurse reviewer can confirm the right label, and add a column REVIEWER_LABEL."*

---
# Section 5 — Incremental AI: only pay for what's new (17 min)

This is the **core technical lesson**. If you recompute AI over every row on every refresh, cost grows with your *history*. With **incremental Dynamic Tables**, AI runs only on **new or changed rows**, so cost grows with your *daily volume*.

### Rules that keep AI pipelines incremental
* Put AI functions in the **SELECT** clause. AI functions in `WHERE`, `GROUP BY`, `HAVING` or `QUALIFY` are not supported for incremental refresh.
* Don't use `CURRENT_TIMESTAMP()` in the SELECT of a Dynamic Table. Keep time-dependent logic (like "hours left on the SLA") in a **view on top** (Section 6).
* Set `REFRESH_MODE = INCREMENTAL` explicitly, so creation **fails loudly** instead of silently falling back to FULL.

> **Cortex Code prompt**
> ```
> Show me the refresh mode of every Dynamic Table in my schema and explain whether
> each one is incremental, and why. Then explain how I keep SILVER_APPEALS incremental
> without a Dynamic Table.
> ```

In [ ]:
%%sql -r dt_modes
SHOW DYNAMIC TABLES IN SCHEMA {{fq}}
->> SELECT "name" AS DYNAMIC_TABLE, "refresh_mode" AS REFRESH_MODE,
           "refresh_mode_reason" AS REFRESH_MODE_REASON, "rows" AS ROW_COUNT
    FROM $1

## 5.1 New appeals arrive
Twenty new appeals were just received. Load them into BRONZE, redact **only the new ones** into SILVER, then refresh the ENRICHED Dynamic Table.

In [ ]:
%%sql -r insert_new_batch
INSERT INTO {{fq}}.BRONZE_APPEALS
SELECT nb.*
FROM {{fq}}.BRONZE_APPEALS_NEW_BATCH nb
WHERE nb.APPEAL_ID NOT IN (SELECT APPEAL_ID FROM {{fq}}.BRONZE_APPEALS)

In [ ]:
%%sql -r load_new_into_silver
-- Incremental load of SILVER: AI_REDACT runs ONLY on appeals not yet in SILVER
INSERT INTO {{fq}}.SILVER_APPEALS
SELECT
    APPEAL_ID, MEMBER_ID, APPEAL_TYPE, APPEAL_RECEIVED_AT, PLAN_DECISION_AT, PLAN_DECISION,
    FORWARDED_TO_IRE, IRE_DECISION, HUMAN_CATEGORY, PLAN_RATIONALE_TEXT,
    redacted AS APPEAL_REASON_REDACTED,
    REGEXP_LIKE(redacted, '.*[1-9][A-Z]{2}[0-9]-?[A-Z]{2}[0-9]-?[A-Z]{2}[0-9]{2}.*')
      OR REGEXP_LIKE(redacted, '.*787-[0-9]{3}-[0-9]{4}.*')
      OR REGEXP_LIKE(redacted, '.*[0-9]{2}/[0-9]{2}/[0-9]{4}.*') AS PHI_LEAK_SUSPECTED
FROM (
    SELECT b.*,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(
            AI_REDACT(b.APPEAL_REASON_TEXT)::STRING,
            '[1-9][A-Z]{2}[0-9]-?[A-Z]{2}[0-9]-?[A-Z]{2}[0-9]{2}', '[MBI]'),
            '787-[0-9]{3}-[0-9]{4}', '[PHONE_NUMBER]'),
            '[0-9]{2}/[0-9]{2}/[0-9]{4}', '[DATE]') AS redacted
    FROM {{fq}}.BRONZE_APPEALS b
    WHERE b.APPEAL_ID NOT IN (SELECT APPEAL_ID FROM {{fq}}.SILVER_APPEALS)
)

In [ ]:
%%sql -r refresh_enriched
ALTER DYNAMIC TABLE {{fq}}.ENRICHED_APPEALS REFRESH

### 5.2 What did each refresh actually process?

In [ ]:
%%sql -r refresh_history
SELECT
    NAME                                        AS DYNAMIC_TABLE,
    REFRESH_ACTION,
    REFRESH_TRIGGER,
    STATE,
    STATISTICS:numInsertedRows::INT             AS ROWS_INSERTED,
    STATISTICS:numDeletedRows::INT              AS ROWS_DELETED,
    DATEDIFF('second', REFRESH_START_TIME, REFRESH_END_TIME) AS DURATION_SECONDS,
    REFRESH_START_TIME
FROM TABLE({{db}}.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME_PREFIX => '{{fq}}.'))
WHERE STATE = 'SUCCEEDED' AND NAME = 'ENRICHED_APPEALS'
ORDER BY REFRESH_START_TIME DESC
LIMIT 5

Read the top row (the refresh you just triggered): `REFRESH_ACTION = INCREMENTAL` and `ROWS_INSERTED = 20`. The initial build processed 300 rows. **This refresh ran AI_CLASSIFY, AI_FILTER and AI_EXTRACT only on the 20 new appeals**, and the SILVER insert ran AI_REDACT only on those same 20.

### Translate that into cost

In [ ]:
%%sql -r incremental_savings
WITH t AS (
    SELECT
        COUNT(*)                                             AS TOTAL_APPEALS,
        COUNT_IF(APPEAL_ID IN (SELECT APPEAL_ID FROM {{fq}}.BRONZE_APPEALS_NEW_BATCH)) AS NEW_APPEALS,
        AVG(AI_COUNT_TOKENS('ai_redact', APPEAL_REASON_TEXT)) AS AVG_TOKENS
    FROM {{fq}}.BRONZE_APPEALS
)
SELECT
    TOTAL_APPEALS,
    NEW_APPEALS,
    TOTAL_APPEALS * 4                           AS AI_CALLS_IF_FULL_REFRESH,
    NEW_APPEALS * 4                             AS AI_CALLS_INCREMENTAL,
    ROUND(100 * (1 - NEW_APPEALS / TOTAL_APPEALS), 1) AS PCT_AI_WORK_AVOIDED,
    ROUND(TOTAL_APPEALS * AVG_TOKENS)           AS REDACT_TOKENS_IF_FULL,
    ROUND(NEW_APPEALS * AVG_TOKENS)             AS REDACT_TOKENS_INCREMENTAL
FROM t

*Four AI calls per appeal: AI_REDACT, AI_CLASSIFY, AI_FILTER and AI_EXTRACT.*

**Scale it in your head:** Triple-S processes thousands of appeals per year. With full refreshes, a daily pipeline re-runs AI over **the entire history every day**. With incremental refreshes, it runs **once per appeal, ever**.

> **Cortex Code prompt:** *"If we receive 40 appeals per day and keep 3 years of history, how many AI calls per year does a daily FULL refresh make compared with INCREMENTAL?"*

---
# Section 6 — GOLD: SLA and IRE overturn metrics (15 min)

> **Cortex Code prompt**
> ```
> Create two GOLD views on ENRICHED_APPEALS:
> 1) GOLD_APPEAL_SLA: SLA hours (72 for EXPEDITED, 720 for STANDARD), due time,
>    and a status: MET, MISSED, OPEN - BREACHED, OPEN - AT RISK (less than 24h left), OPEN - ON TRACK
> 2) GOLD_IRE_OVERTURN: for appeals the IRE has decided, the overturn rate by
>    AI_CATEGORY and IS_WEAK_RATIONALE.
> Then chart the overturn rate by category, weak vs. strong rationale.
> ```

In [ ]:
%%sql -r create_gold_sla
CREATE OR REPLACE VIEW {{fq}}.GOLD_APPEAL_SLA
COMMENT = 'Regulatory SLA status for every appeal (72h expedited / 30 days standard)'
AS
SELECT
    APPEAL_ID, APPEAL_TYPE, AI_CATEGORY, APPEAL_RECEIVED_AT, PLAN_DECISION_AT, PLAN_DECISION,
    IFF(APPEAL_TYPE = 'EXPEDITED', 72, 720)                    AS SLA_HOURS,
    DATEADD(hour, SLA_HOURS, APPEAL_RECEIVED_AT)               AS SLA_DUE_AT,
    DATEDIFF(hour, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, SLA_DUE_AT) AS HOURS_REMAINING,
    CASE
        WHEN PLAN_DECISION_AT IS NOT NULL AND PLAN_DECISION_AT <= SLA_DUE_AT THEN 'MET'
        WHEN PLAN_DECISION_AT IS NOT NULL                                  THEN 'MISSED'
        WHEN HOURS_REMAINING < 0                                           THEN 'OPEN - BREACHED'
        WHEN HOURS_REMAINING < 24                                          THEN 'OPEN - AT RISK'
        ELSE 'OPEN - ON TRACK'
    END AS SLA_STATUS
FROM {{fq}}.ENRICHED_APPEALS

In [ ]:
%%sql -r create_gold_overturn
CREATE OR REPLACE VIEW {{fq}}.GOLD_IRE_OVERTURN
COMMENT = 'IRE overturn rate by AI appeal category and rationale strength'
AS
SELECT
    AI_CATEGORY,
    IFF(IS_WEAK_RATIONALE, 'Weak rationale', 'Specific rationale') AS RATIONALE_STRENGTH,
    COUNT(*)                                    AS IRE_DECIDED,
    COUNT_IF(IRE_DECISION = 'OVERTURNED')       AS IRE_OVERTURNED,
    ROUND(100 * IRE_OVERTURNED / IRE_DECIDED, 1) AS OVERTURN_RATE_PCT
FROM {{fq}}.ENRICHED_APPEALS
WHERE IRE_DECISION IN ('OVERTURNED', 'UPHELD')
GROUP BY 1, 2

In [ ]:
%%sql -r sla_summary
SELECT APPEAL_TYPE, SLA_STATUS, COUNT(*) AS APPEALS
FROM {{fq}}.GOLD_APPEAL_SLA
GROUP BY 1, 2
ORDER BY 1, 2

In [ ]:
%%sql -r at_risk_now
-- The work queue for today: open appeals that are breached or will breach within 24h
SELECT APPEAL_ID, APPEAL_TYPE, AI_CATEGORY, APPEAL_RECEIVED_AT, SLA_DUE_AT, HOURS_REMAINING, SLA_STATUS
FROM {{fq}}.GOLD_APPEAL_SLA
WHERE SLA_STATUS IN ('OPEN - BREACHED', 'OPEN - AT RISK')
ORDER BY HOURS_REMAINING

In [ ]:
%%sql -r overturn_overall
SELECT RATIONALE_STRENGTH, SUM(IRE_DECIDED) AS IRE_DECIDED, SUM(IRE_OVERTURNED) AS IRE_OVERTURNED,
       ROUND(100 * SUM(IRE_OVERTURNED) / SUM(IRE_DECIDED), 1) AS OVERTURN_RATE_PCT
FROM {{fq}}.GOLD_IRE_OVERTURN
GROUP BY 1
ORDER BY 1

In [ ]:
%%sql -r overturn_by_category
SELECT * FROM {{fq}}.GOLD_IRE_OVERTURN

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = overturn_by_category.to_pandas() if not isinstance(overturn_by_category, pd.DataFrame) else overturn_by_category.copy()
pivot = df.pivot_table(index="AI_CATEGORY", columns="RATIONALE_STRENGTH", values="OVERTURN_RATE_PCT", fill_value=0)

ax = pivot.plot(kind="barh", figsize=(10, 6), color={"Weak rationale": "#d62728", "Specific rationale": "#1f77b4"})
ax.set_title("IRE overturn rate by appeal category: weak vs. specific denial rationale")
ax.set_xlabel("IRE overturn rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### The insight
Denials with a **weak, generic rationale** (*"Denied per plan policy."*) are overturned by the IRE **several times more often** than denials that cite a specific criterion. That is an actionable finding for Utilization Management:

* Flag weak rationales **before** the denial letter goes out (`AI_FILTER` already does it, incrementally)
* Coach reviewers in the categories with the highest overturn rate
* Track the metric weekly, because the GOLD views are always current

---
# Wrap-up

## What you built in 90 minutes
| Layer | Object | AI inside |
|---|---|---|
| BRONZE | `BRONZE_APPEALS` + masking policies | — |
| SILVER | `SILVER_APPEALS` (table, loaded incrementally) | AI_REDACT + regex safety net + leak detection |
| ENRICHED | `ENRICHED_APPEALS` (incremental DT) | AI_CLASSIFY, AI_FILTER, AI_EXTRACT |
| GOLD | `GOLD_APPEAL_SLA`, `GOLD_IRE_OVERTURN` | — |

## The Cortex Code workflow you practiced
1. **Context:** describe the tables, the PHI and the goal
2. **Estimate:** count tokens before spending
3. **Prototype:** 5 to 10 rows, side by side
4. **Validate:** leak report, agreement rate, confusion matrix
5. **Productionize:** incremental Dynamic Tables
6. **Explain:** ask Cortex Code why a table is or isn't incremental

## Take-home challenges (with Cortex Code)
1. *"Build a semantic view over GOLD_APPEAL_SLA and GOLD_IRE_OVERTURN and a Cortex Agent so Appeals managers can ask 'Which expedited appeals are at risk today?' in CoWork."*
2. *"Create a Snowflake alert that emails the appeals team whenever an expedited appeal has less than 12 hours left."*
3. *"Route rows with PHI_LEAK_SUSPECTED to a quarantine table and exclude them from ENRICHED_APPEALS."*
4. *"Add AI_SENTIMENT on the member text and check whether frustrated members appeal more often to the IRE."*

---
## Teardown (run only after the lab)
Commented out on purpose.

In [ ]:
# session.sql(f"DROP DYNAMIC TABLE IF EXISTS {fq}.ENRICHED_APPEALS").collect()
# session.sql(f"DROP DYNAMIC TABLE IF EXISTS {fq}.SILVER_APPEALS").collect()
# session.sql(f"DROP SCHEMA IF EXISTS {fq}").collect()
# print(f"Dropped {fq}")